# Lekcja 01 - Wprowadzenie do Agentów AI

Witamy w pierwszej lekcji kursu **Agenci AI dla początkujących**!

**Agent AI** to program, który wykorzystuje duży model językowy (LLM) jako swój silnik rozumowania i może podejmować *działania* w rzeczywistym świecie — wywoływać API, zapytywać bazy danych lub uruchamiać kod — aby osiągnąć cel w imieniu użytkownika.

W tym notatniku zbudujesz swojego pierwszego agenta: **Agenta Podróży**, który poleca miejsca na wakacje. Po drodze nauczysz się, jak:

1. Połączyć się z usługą Azure AI Foundry Agent za pomocą **Microsoft Agent Framework**.
2. Wyposażyć agenta w **narzędzie** — prostą funkcję Pythona, którą może wywołać.
3. Uruchomić agenta i przeanalizować jego odpowiedź.
4. Transmitować odpowiedź agenta token po tokenie.


## Konfiguracja

Przed uruchomieniem tego notatnika upewnij się, że masz:

1. **Projekt Azure AI Foundry** z wdrożonym modelem czatu (np. `gpt-4o-mini`).
2. **Zalogowano się za pomocą Azure CLI** — uruchom `az login` w terminalu.
3. **Ustawione wymagane zmienne środowiskowe:**
   - `AZURE_AI_PROJECT_ENDPOINT` — punkt końcowy Twojego projektu Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nazwa wdrożonego modelu.

Poniższa komórka instaluje potrzebne pakiety Pythona.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tworzenie Twojego Pierwszego Agenta

Agent potrzebuje dwóch rzeczy:

- **Instrukcji**, które mówią mu *kim jest* i *jak się zachowywać* (polecenie systemowe).
- **Narzędzi** — funkcji Pythona oznaczonych dekoratorem `@tool`, które agent może wywołać, aby uzyskać informacje lub wykonać działania.

Poniżej definiujemy proste narzędzie, które zwraca listę popularnych miejsc wakacyjnych. Agent będzie korzystał z tego narzędzia, gdy użytkownik poprosi o rekomendacje podróży.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Odpowiedzi strumieniowe

Dla bardziej interaktywnego doświadczenia możesz **strumieniowo** odbierać odpowiedź agenta. Zamiast czekać na pełną odpowiedź, agent dostarcza fragmenty tekstu w miarę ich generowania. Jest to szczególnie przydatne w interfejsach czatu, gdzie chcesz wyświetlać wynik w czasie rzeczywistym.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

- **Utworzyć dostawcę** łączącego się z usługą Azure AI Foundry Agent za pomocą `AzureAIProjectAgentProvider`.
- **Zdefiniować narzędzie** używając dekoratora `@tool`, aby agent mógł wywoływać Twoje funkcje Pythona.
- **Uruchomić agenta** z wiadomością od użytkownika i wydrukować jego odpowiedź.
- **Strumieniować odpowiedzi** dla wyjścia w czasie rzeczywistym.

W następnej lekcji bardziej szczegółowo omówimy ramy agentyczne i nauczymy się, jak wyposażyć agentów w potężniejsze narzędzia oraz możliwości rozumowania wieloetapowego.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Niniejszy dokument został przetłumaczony przy użyciu automatycznej usługi tłumaczeniowej AI [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy wszelkich starań, aby tłumaczenie było jak najbardziej poprawne, prosimy mieć na uwadze, że przekłady automatyczne mogą zawierać błędy lub niedokładności. Oryginalny dokument w języku źródłowym powinien być traktowany jako źródło nadrzędne. W przypadku informacji o znaczeniu krytycznym zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za ewentualne nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
